<a href="https://colab.research.google.com/github/sportcman/GPT/blob/main/07_01_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

In [ ]:
pip install transformers[torch]

In [ ]:
pip install torch_xla

In [ ]:
pip install accelerate -U

In [ ]:
pip install torch torchvision torchaudio

**`Создание модели. С малым количеством настроек параметров.`**

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

# Импорт необходимых классов из библиотеки Transformers
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

# Создаем конфигурацию модели с большими параметрами
model_config = GPT2Config.from_pretrained('ai-forever/rugpt2large')  # Загрузка предварительно обученной конфигурации модели GPT-2.
model_config.n_layer = 64  # Установка числа слоев модели, это позволит модели обнаруживать более сложные зависимости в данных.
model_config.n_head = 512  # Установка числа attention-головок модели, это помогает модели обрабатывать различные аспекты контекста одновременно.
model_config.n_embd = 512  # Установка размерности векторного представления (embedding), более высокая размерность позволяет модели учиться более сложным взаимосвязям между словами.
model_config.intermediate_size = 512  # Установка размера промежуточного слоя модели, более высокое значение может помочь модели обрабатывать более сложные взаимосвязи между словами.
model_config.hidden_size = 512  # Установка размера скрытого состояния модели,чтобы увеличить количество параметров и, следовательно, емкость модели, большая емкость может помочь модели представлять более сложные зависимости в данных.
model_config.mem_len = 512  # Установка длины оперативной памяти модели, Более длинный контекст может повысить качество и разнообразие сгенерированного текста.

# Создаем модель на основе заданной конфигурации
model = GPT2LMHeadModel(config=model_config)  # Создание объекта модели GPT-2 с головой для языкового моделирования на основе заданной конфигурации

# Создаем токенизатор
tokenizer = GPT2Tokenizer.from_pretrained('ai-forever/rugpt2large')  # Загрузка предварительно обученного токенизатора GPT-2

# Сохраняем модель и токенизатор
model.save_pretrained('/gdrive/MyDrive/TrenerGpt/model')  # Сохранение модели в указанной директории
tokenizer.save_pretrained('/gdrive/MyDrive/TrenerGpt/model')  # Сохранение токенизатора в указанной директории


***`Создание модели. С большим количеством настроек параметров.`***

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

# Создаем конфигурацию модели с малыми параметрами
model_config = GPT2Config(
    vocab_size=50257,   # Размер словаря модели
    n_positions=1024,   # Максимальное количество позиций в последовательности
    n_ctx=1024,         # Размер контекста (максимальная длина входной последовательности)
    n_embd=256,         # Размерность эмбеддинга
    n_layer=4,          # Количество слоев модели
    n_head=4,            # Количество голов в слоях AufioReg
    intermediate_size=1024,  # Размер промежуточного слоя в блоке
    hidden_size=256,         # Размер скрытого состояния
    num_labels=2        # Количество меток задачи
)

# Создаем модель на основе заданной конфигурации
model = GPT2LMHeadModel(config=model_config)

# Создаем русский токенизатор
tokenizer = GPT2Tokenizer.from_pretrained('sberbank-ai/rugpt3small_based_on_gpt2')

# Сохраняем модель и токенизатор
model.save_pretrained('/gdrive/MyDrive/TrenerGpt/model')
tokenizer.save_pretrained('/gdrive/MyDrive/TrenerGpt/model')

Обучение МОДЕЛИ.

In [ ]:
from transformers import GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments, GPT2LMHeadModel
import argparse
import torch
import torch_xla.core.xla_model as xm

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--num_epochs', type=int, default=3)
    parser.add_argument('--dataset_path', type=str, default="./dataset.txt")
    parser.add_argument('--device', type=str, default="cpu")
    _, unknown_args = parser.parse_known_args()

    args = {}
    for arg in unknown_args:
        key_value = arg.split('=')
        if len(key_value) == 2:
            args[key_value[0][2:]] = key_value[1]

    dataset_path = args.get('dataset_path', '/gdrive/MyDrive/TrenerGpt/dataset.txt')
    model_path = "/gdrive/MyDrive/TrenerGpt/model"
    num_epochs = int(args.get('num_epochs', 3))
    batch_size = 4

    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    model = GPT2LMHeadModel.from_pretrained(model_path)

    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=dataset_path,
        block_size=128
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    device = args.get('device', 'cpu')
    if device == "gpu":
        device = torch.device("cuda")
    elif device == "tpu":
        device = xm.xla_device()
    else:
        device = torch.device("cpu")

    training_args = TrainingArguments(
        output_dir=model_path,
        overwrite_output_dir=True,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        save_total_limit=1,
        dataloader_num_workers=2,  # Уменьшение количества рабочих процессов до 2
        gradient_accumulation_steps=1,
        report_to="tensorboard",
    )

    if device == xm.xla_device():
        training_args.__dict__["_n_gpu"] = 1

    trainer = Trainer(
        model=model.to(device),
        args=training_args,
        data_collator=data_collator,
        train_dataset=dataset,
    )

    if device == xm.xla_device():
        trainer.train_device = xm.xla_device()
        trainer.accelerator = "xla"

    trainer.train()
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)

if __name__ == '__main__':
    main()

***Скрипт для проверки ответов.***

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import subprocess

class GPT2Generator:
    def __init__(self, model_path):
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        self.model = GPT2LMHeadModel.from_pretrained(model_path)

    def generate_text(self, input_text, temperature_value, length_value, num_results, no_repeat_ngram_size):
        input_ids = self.tokenizer.encode(input_text, return_tensors='pt')
        attention_mask = torch.ones(input_ids.shape, dtype=torch.long, device=input_ids.device)

        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=length_value,
            num_return_sequences=num_results,
            no_repeat_ngram_size=no_repeat_ngram_size,
            repetition_penalty=1.5,
            temperature=temperature_value,
            do_sample=True
        )

        result_text = ""
        for i, output in enumerate(outputs):
            generated_text = self.tokenizer.decode(output, skip_special_tokens=True)
            result_text += f"Результат {i+1}:\n{generated_text}\n\n"

        return result_text

gpt2_generator = GPT2Generator("/gdrive/MyDrive/TrenerGpt/model")
temperature_value = 0.1
length_value = 70
num_results = 1
ngram_value = 2

def generate_text():
    input_text = input("Введи затравку: ")
    result_text = gpt2_generator.generate_text(input_text, temperature_value, length_value, num_results, ngram_value)
    print(result_text)


if __name__ == "__main__":
    while True:
        user_input = input("Выберите действие (1 - сгенерировать текст, 2 - выход): ")
        if user_input == "1":
            generate_text()
        elif user_input == "2":
            break
        else:
            print("Некорректный ввод. Попробуйте снова.")